In [1]:
from copy import deepcopy
from pathlib import Path

import wandb

from conf.behavior_cloning.diffusion.five_demos.default import config
from tapas_gmm.behavior_cloning import run_training
from tapas_gmm.dataset.bc import BCDataset
from tapas_gmm.dataset.scene import SceneDataset
from tapas_gmm.encoder.encoder import ObservationEncoderConfig
from tapas_gmm.policy.diffusion import DiffusionPolicy, DiffusionPolicyTrainingConfig

2026-07-17 17:03:00.447 | INFO     |  Running on cpu


In [2]:
data_root = Path("../outputs/bimanual_dataset")
horizon = 16
n_obs_steps = 2
n_action_steps = 8
epochs = 20

leader_config = deepcopy(config)
follower_config = deepcopy(config)

In [3]:
for policy_config in (leader_config.policy, follower_config.policy):
    policy_config.action_dim = 8
    policy_config.obs_dim = 35
    policy_config.horizon = horizon
    policy_config.n_obs_steps = n_obs_steps
    policy_config.n_action_steps = n_action_steps
    policy_config.training = DiffusionPolicyTrainingConfig(lr_num_epochs=epochs)
    policy_config.obs_encoder = ObservationEncoderConfig(
        ee_pose=True,
        object_poses=True,
    )
    policy_config.unet.input_dim = 8
    policy_config.unet.down_dims = (64, 128, 256)

leader_config.policy.arm = "left"
leader_config.policy.unet.global_cond_dim = 35 * n_obs_steps

follower_config.policy.arm = "right"
follower_config.policy.condition_on_arm = "left"
follower_config.policy.unet.global_cond_dim = 35 * n_obs_steps + horizon * 8

In [4]:
for training_config in (leader_config, follower_config):
    training_config.bc_data.fragment_length = horizon + 1
    training_config.bc_data.pre_padding = n_obs_steps - 1
    training_config.bc_data.post_padding = n_action_steps - 1
    training_config.bc_data.cameras = tuple()
    training_config.training.epochs = epochs
    training_config.training.eval_freq = 5
    training_config.data_loader.batch_size = 8
    training_config.data_loader.eval_batchsize = 8

In [5]:
loaded_dataset = SceneDataset(data_root=data_root)

bc_dataset = BCDataset(
    scene_dataset=loaded_dataset,
    config=leader_config.bc_data,
)

leader_policy = DiffusionPolicy(leader_config.policy)
follower_policy = DiffusionPolicy(follower_config.policy)

2026-07-17 17:03:02.033 | INFO     |  Initializing datasete using ../outputs/bimanual_dataset/metadata.json
2026-07-17 17:03:02.037 | INFO     |  Extracted gt object labels []
2026-07-17 17:03:02.038 | INFO     |  Extracted tsdf object labels []
2026-07-17 17:03:02.038 | INFO     |  Initializing BCDataset:
2026-07-17 17:03:02.038 | INFO     |    Training on fragments of length 17.
2026-07-17 17:03:02.039 | INFO     |    Loading raw data for encoder.
2026-07-17 17:03:02.039 | INFO     |  Initializing DiffusionPolicy:
2026-07-17 17:03:02.039 | INFO     |    Initializing Policy:
2026-07-17 17:03:02.209 | INFO     |    number of parameters: 5522376
2026-07-17 17:03:02.212 | INFO     |    No encoder config provided. Using None.
None
2026-07-17 17:03:02.381 | INFO     |    number of parameters: 5981128
None


In [6]:
wandb.init(mode="disabled")

run_training(
    leader_policy,
    bc_dataset,
    leader_config,
    "../outputs/diffusion_leader_left",
)

leader_policy.to_disk("../outputs/diffusion_leader_left.pt")

2026-07-17 17:03:02.633 | INFO     |  No datasplit specified.
2026-07-17 17:03:02.722 | INFO     |    Setting action scaling for optimal policy performance. Using DP normalizer implementation.
2026-07-17 17:03:04.086 | INFO     |  Beginning training.


  0%|          | 0/20 [00:00<?, ?it/s]

2026-07-17 17:32:59.699 | INFO     |  Saving policy at ../outputs/diffusion_leader_left.pt


In [7]:
run_training(
    follower_policy,
    bc_dataset,
    follower_config,
    "../outputs/diffusion_follower_right",
)

follower_policy.to_disk("../outputs/diffusion_follower_right.pt")

  0%|          | 0/20 [00:00<?, ?it/s]

2026-07-17 18:16:19.426 | INFO     |  Saving policy at ../outputs/diffusion_follower_right.pt
